# MIST 담당 2 · Colab 실행 노트북

이 노트북은 프로젝트 코드를 Colab에서 실행하고, 대용량 데이터·산출물은 Google Drive의 `tobigs_2026`에 보관한다.

## 실행 순서
1. 런타임을 GPU로 설정한다.
2. Drive를 마운트한다.
3. 경로 점검 셀에서 실제 데이터 위치를 확인한다.
4. `jiye` 브랜치를 clone 또는 갱신한다.
5. 환경 설치와 데이터 검증을 수행한다.
6. 승인된 실험 셀만 순서대로 실행한다.

> 주의: `calib`, `meta`, `test`는 모델 학습에 사용하지 않는다. 데이터·outputs·artifacts·checkpoints는 Git에 올리지 않는다.

In [ ]:
# 1. GPU 런타임 확인
import os
import subprocess

print('COLAB_GPU =', os.environ.get('COLAB_GPU', '없음'))
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout.splitlines()[0])
else:
    print('GPU가 보이지 않습니다. 런타임 > 런타임 유형 변경에서 GPU를 선택하세요.')

In [ ]:
# 2. Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. 공통 경로 설정 — Drive 구조가 다르면 DRIVE_ROOT만 수정
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/tobigs_2026')
DATA_DIR = DRIVE_ROOT / 'data' / 'processed'
OUTPUT_DIR = DRIVE_ROOT / 'outputs'
ARTIFACT_DIR = DRIVE_ROOT / 'artifacts'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
REPO_DIR = Path('/content/molecular-reliability-signals')

for path in [DRIVE_ROOT, OUTPUT_DIR, ARTIFACT_DIR, CHECKPOINT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('Drive root :', DRIVE_ROOT)
print('Data dir   :', DATA_DIR, '존재:', DATA_DIR.exists())
print('Repo dir   :', REPO_DIR)
assert DATA_DIR.exists(), f'데이터 폴더가 없습니다: {DATA_DIR}'

## GitHub 코드 가져오기

아래 설정 셀의 작업 브랜치를 checkout하고 fast-forward 방식으로 갱신한다. 개인 토큰을 노트북에 적지 않는다.

In [ ]:
# 4. 프로젝트 작업 브랜치 clone/갱신
import subprocess

REPO_URL = 'https://github.com/Feellived/molecular-reliability-signals.git'
BRANCH = 'jiye'

if not (REPO_DIR / '.git').exists():
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)

branch = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'branch', '--show-current'],
    check=True, capture_output=True, text=True,
).stdout.strip()
assert branch == BRANCH, f'현재 브랜치가 {BRANCH}가 아닙니다: {branch}'
print('현재 작업 브랜치:', branch)

In [ ]:
# 5. 현재 단계의 로컬 지문 모델 의존성 설치
%pip install -q rdkit pandas numpy scipy scikit-learn xgboost pyarrow pyyaml joblib pytest

import rdkit, sklearn, xgboost
print('RDKit       :', rdkit.__version__)
print('scikit-learn:', sklearn.__version__)
print('XGBoost     :', xgboost.__version__)

In [ ]:
# 6. Drive 데이터가 22종인지 먼저 확인
split_files = sorted(DATA_DIR.glob('*/splits.csv'))
print('splits.csv 수:', len(split_files))
print([path.parent.name for path in split_files])
assert len(split_files) == 22, f'22개가 필요하지만 {len(split_files)}개를 찾았습니다.'

In [ ]:
# 7. AqSolDB의 legacy N-oxide SMILES 2개 교정
# 원본 CSV 백업과 감사 보고서가 Drive에 자동으로 생성된다.
repair_command = [
    'python', str(REPO_DIR / 'scripts' / 'repair_solubility_n_oxide.py'),
    '--processed-dir', str(DATA_DIR),
    '--apply',
]
completed = subprocess.run(repair_command, cwd=REPO_DIR)
assert completed.returncode == 0, 'SMILES 교정 실패'
print('교정 감사 보고서:', DATA_DIR / '_audit' / 'role2' / 'reports' / 'smiles_repair.json')

In [ ]:
# 8. 전체 데이터 검증 — 모델 학습 전에 반드시 통과해야 함
validation_report = DATA_DIR / '_audit' / 'role2' / 'reports' / 'validation_colab.json'
validation_report.parent.mkdir(parents=True, exist_ok=True)
command = [
    'python', str(REPO_DIR / 'scripts' / 'validate_processed_data.py'),
    '--processed-dir', str(DATA_DIR),
    '--report', str(validation_report),
]
completed = subprocess.run(command, cwd=REPO_DIR)
assert completed.returncode == 0, '데이터 검증 실패: 위 오류와 보고서를 확인하세요.'
print('검증 보고서:', validation_report)

## 지문 모델 파일럿 재현

아래 셀은 `dili`의 Morgan + Random Forest/XGBoost 파일럿을 Colab에서 재현한다. ChemBERTa GPU 학습은 지문 모델 파이프라인 검증 후 별도 단계에서 수행한다.

In [ ]:
# 9. dili 지문 모델 파일럿
command = [
    'python', str(REPO_DIR / 'scripts' / 'run_fingerprint_dataset.py'),
    '--dataset', 'dili',
    '--processed-dir', str(DATA_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--artifact-dir', str(ARTIFACT_DIR),
]
completed = subprocess.run(command, cwd=REPO_DIR)
assert completed.returncode == 0, 'dili 파일럿 실행 실패'
print('예측:', OUTPUT_DIR / 'dili' / 'fingerprint_predictions.csv')
print('지문:', ARTIFACT_DIR / 'dili' / 'morgan_r2_2048_chiral.npz')

In [ ]:
# 10. 결과 파일 최소 검증
import pandas as pd

prediction_path = OUTPUT_DIR / 'dili' / 'fingerprint_predictions.csv'
predictions = pd.read_csv(prediction_path)
assert len(predictions) == 475
assert predictions['row_uid'].is_unique
assert predictions[['pred_rf', 'pred_xgb', 'pred_fp_primary']].notna().all().all()
display(predictions.head())
print('primary model:', predictions['fp_primary_model'].unique().tolist())

## 지문 모델 22종 배치 실행

아래 셀의 `RUN_FULL_BATCH`를 `True`로 바꿔 실행한다. 각 물성이 끝날 때마다 Drive에 결과와 manifest가 저장된다. 런타임이 끊기면 같은 셀을 다시 실행하면 완료된 물성은 자동으로 건너뛴다.

In [ ]:
# 11. 22종 전체 RF/XGBoost 실행
RUN_FULL_BATCH = False  # 실행할 때만 True로 변경

if RUN_FULL_BATCH:
    batch_command = [
        'python', str(REPO_DIR / 'scripts' / 'run_fingerprint_batch.py'),
        '--processed-dir', str(DATA_DIR),
        '--output-dir', str(OUTPUT_DIR),
        '--artifact-dir', str(ARTIFACT_DIR),
    ]
    completed = subprocess.run(batch_command, cwd=REPO_DIR)
    print('batch return code:', completed.returncode)
else:
    print('실행 준비 상태입니다. RUN_FULL_BATCH=True로 바꾸면 시작합니다.')

In [ ]:
# 12. 배치 진행 상황 확인
import json

manifest_path = OUTPUT_DIR / '_batch' / 'fingerprint_manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    status = pd.DataFrame.from_dict(manifest.get('datasets', {}), orient='index')
    display(status)
    print('완료/건너뜀:', int(status['status'].isin(['complete', 'skipped_complete']).sum()))
    print('실패:', int(status['status'].eq('failed').sum()))
else:
    print('아직 배치를 실행하지 않았습니다.')

## 후속 실험 단계

`dili` 재현과 결과 검증을 완료한 뒤 다음 실험을 순서대로 추가한다.

- 회귀 지문 모델 파일럿
- 지문 모델 22종 전체 실행
- ChemBERTa 정규 학습
- ChemBERTa 증강 학습
- 컨포멀 보정과 모델 불일치 계산